# Chronos-2 energy fine-tune (GPU)

Generic fine-tune of `amazon/chronos-2` on the public UCI Appliances
Energy dataset via the `chronos` recipe in
powerdecksystems/powerdeck-models. Kaggle supplies the GPU
(~30 h/week free); the workflow in-repo covers CPU-feasible recipes.

Secrets needed in this kernel (Kaggle → Add-ons → Secrets):
`GITHUB_TOKEN` — classic PAT, `public_repo` + `workflow` on
powerdecksystems. `HF_TOKEN` optional.

In [ ]:
!pip install -q --index-url https://download.pytorch.org/whl/cpu torch || true
!pip install -q transformers onnx onnxruntime chronos-forecasting huggingface_hub pandas
!git clone --depth 1 https://github.com/powerdecksystems/powerdeck-models /kaggle/working/pd-models
import os; os.chdir('/kaggle/working/pd-models')

In [ ]:
from kaggle_secrets import UserSecretsClient
os.environ['GITHUB_TOKEN'] = UserSecretsClient().get_secret('GITHUB_TOKEN')
NAME = 'chronos-2-energy'
TAG  = 'models-finetune'
STEPS = '500'   # LoRA steps — bounded so a run stays well inside quota

In [ ]:
!python tools/train.py --recipe chronos --name {NAME} --model amazon/chronos-2 \
    --out dist/{NAME} --epochs 0 --steps {STEPS} \
    --title 'Chronos-2 (energy fine-tune)' \
    --description 'Chronos-2 LoRA fine-tuned on UCI appliances energy data' \
    --use-case 'Energy-domain forecasting' --tier ai --kind forecast

In [ ]:
# publish: release asset + manifest commit (same contract as train.yml)
!curl -sL https://github.com/cli/cli/releases/download/v2.65.0/gh_2.65.0_linux_amd64.tar.gz | tar xz
GH='/kaggle/working/gh_2.65.0_linux_amd64/bin/gh'
os.environ['GH_TOKEN'] = os.environ['GITHUB_TOKEN']
import subprocess
repo = 'powerdecksystems/powerdeck-models'
subprocess.run([GH,'release','view',TAG,'--repo',repo], capture_output=True).returncode \
  and subprocess.check_call([GH,'release','create',TAG,'--repo',repo,'--title',TAG])
subprocess.check_call([GH,'release','upload',TAG,f'dist/{NAME}/model.onnx',
                       '--repo',repo,'--clobber'])
subprocess.check_call(['python','tools/manifest_update.py','--fragment',
                       f'dist/{NAME}/fragment.json','--manifest','manifest.json',
                       '--repo',repo,'--tag',TAG])
subprocess.check_call(['git','config','user.name','powerdeck-models-bot'])
subprocess.check_call(['git','config','user.email','bot@powerdecksystems.github'])
subprocess.check_call(['git','add','manifest.json'])
subprocess.check_call(['git','commit','-m',f'manifest: {NAME} @ {TAG} (kaggle)'])
subprocess.check_call(['git','push'])